In [1]:

import re
import random
import numpy as np
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load Qwen2.5 with 0.5 billion parameters.
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Loading {MODEL_ID} on {device.upper()}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device)
model.eval()
print("Local LLM loaded and ready.")


def generate_local_llm(
    system_prompt: str,
    user_prompt: str,
    max_new_tokens: int = 350,
    deterministic: bool = False
) -> str:
    """Run inference locally using the Hugging Face chat template."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if deterministic:
        gen_kwargs["do_sample"] = False  # Greedy decoding reduces RAG hallucinations
    else:
        gen_kwargs.update({
            "do_sample": True,
            "temperature": 0.3,
            "top_p": 0.9
        })

    with torch.inference_mode():
        outputs = model.generate(**inputs, **gen_kwargs)

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

Loading Qwen/Qwen2.5-0.5B-Instruct on CPU...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Local LLM loaded and ready.


In [2]:
# Regex for PII masking
PII_MASKING_RULES = [
    (
        "Aadhaar / 12-Digit ID",
        re.compile(r"(?<!\d)\d{4}[\s-]?\d{4}[\s-]?\d{4}(?!\d)"),
        "****-****-****"
    ),
    (
        "PAN Card",
        re.compile(r"\b[A-Z]{5}\d{4}[A-Z]\b"),
        "**********"
    ),
    (
        "Email Address",
        re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"),
        "***@***.***"
    ),
    (
        "Phone Number",
        re.compile(r"(?<!\d)(?:\+91[\s-]?|0)?[6-9]\d{4}[\s-]?\d{5}(?!\d)"),
        "**********"
    ),
    (
        "Home Address",
        re.compile(
            r"(?i)\b(?:R/o|Resident\s+of|H\.?\s*No\.?|House\s+No\.?|Flat\s+No\.?|Plot\s+No\.?)"
            r"[\s:,-]*[^.\n;]{4,90}"
        ),
        "[ADDRESS: ********]"
    ),
    (
        "PIN Code",
        re.compile(r"(?i)\b(?:PIN|Pin\s*Code)[\s:-]*[1-9]\d{5}\b"),
        "PIN: ******"
    ),
]


def mask_pii_asterisks(text: str) -> tuple[str, str]:
    """
    Replaces citizen PII with asterisk masks BEFORE the text is passed to the LLM.
    Returns:
        masked_text (str): Sanitized text safe for LLM inference.
        audit_summary (str): Human-readable log of masked PII items.
    """
    masked_text = str(text)
    redacted_counts = {}

    for label, pattern, mask_token in PII_MASKING_RULES:
        matches = pattern.findall(masked_text)
        if matches:
            redacted_counts[label] = len(matches)
            masked_text = pattern.sub(mask_token, masked_text)

    if redacted_counts:
        details = ", ".join(f"{k}: {v}" for k, v in redacted_counts.items())
        audit_summary = f"PII Masked Before LLM Call -> {details}"
    else:
        audit_summary = "No PII detected. Clean text forwarded to LLM."

    return masked_text, audit_summary

print("Asterisk PII masking gate initialized.")

Asterisk PII masking gate initialized.


In [3]:
# ---------------------------------------------------------
# TASK 3.1: C.R.O.S.S. System Prompt
# ---------------------------------------------------------
CROSS_SYSTEM_PROMPT = """
[C - CONTEXT]
You are operating inside the CPGRAMS / JK-IGRAMS Grievance Redressal Co-Pilot for District Administration Kulgam, Jammu & Kashmir.
 Incoming citizen complaints have already passed through a PII privacy filter that replaces personal IDs, phone numbers, and addresses with asterisks (e.g., ****-****-****).

[R - ROLE]
Act as a senior, empathetic, and impartial e-Governance Grievance Redressal Officer assisting the Deputy Commissioner's Office, District Kulgam.

[O - OBJECTIVE]
Read the sanitized citizen complaint and output exactly three clearly labeled sections:
### 1. Neutral Summary
Summarize the factual issue in exactly 2 neutral, objective sentences (strip out anger and insults).

### 2. Sentiment & Urgency Score
- **Citizen Sentiment:** (e.g., Angry, Frustrated, Anxious)
- **Urgency Level:** (Strictly one of: High, Medium, or Low)
- **Reason:** One concise sentence justifying the urgency level.

### 3. Draft Acknowledgment Letter
Write a polite, empathetic, formal acknowledgment letter confirming registration and forwarding to the concerned department
like Deputy Commissioner, ADC,ACR,ACD,Chief Medical Officer,Tehsildar,BDO,Executive Engineer PWD, Executive Engineer Jal shakti,Department Heads etc in District Kulgam for time-bound verification.

[S - STYLE]
Calm, respectful, citizen-centric, formal official tone.

[S - SPECIFICS]
- NEVER argue with the citizen or blame any official before inquiry.
- NEVER invent fake dates, complaint numbers, or financial promises.
- Keep all masked placeholders (****) redacted; do not ask the citizen to re-type sensitive IDs in plain text.
- Keep the Neutral Summary strictly to 2 sentences and the letter under 120 words.
""".strip()


def analyze_grievance_with_pii_masking(raw_complaint: str):
    """Masks PII first, then sends ONLY the masked text to the local LLM."""
    if not raw_complaint or not raw_complaint.strip():
        return "", "Waiting for input...", "Please enter a citizen complaint to analyze."

    # STEP 1: Mandatory PII Masking BEFORE LLM sees the text
    sanitized_text, audit_summary = mask_pii_asterisks(raw_complaint.strip())

    # STEP 2: Pass ONLY sanitized_text to the LLM
    user_prompt = f'Sanitized Citizen Complaint:\n"""\n{sanitized_text}\n"""'
    llm_response = generate_local_llm(
        system_prompt=CROSS_SYSTEM_PROMPT,
        user_prompt=user_prompt,
        max_new_tokens=380,
        deterministic=False
    )

    return sanitized_text, audit_summary, llm_response


# ---------------------------------------------------------
# TASK 3.2: Mini-RAG Government SOP Knowledge Base
# ---------------------------------------------------------
SOP_KNOWLEDGE_BASE = [
    {
        "sop_id": "SOP-KPDCL-2025-01",
        "title": "KPDCL Distribution Transformer Repair & Replacement SOP (District Kulgam)",
        "content": (
            "1. Reporting: Damage to a distribution transformer (DT) must be reported to the concerned "
            "Assistant Executive Engineer (AEE) Electric Sub-Division or via JK-IGRAMS / CPGRAMS.\n"
            "2. Replacement Timelines (SLA): Under J&K Electricity Supply Code norms, damaged transformers "
            "must be replaced from buffer stock within 24 hours in urban/municipal areas (Kulgam, Yaripora, "
            "Frisal, Qaimoh towns) and within 48 hours in rural/far-flung areas (including D.H. Pora and Kund).\n"
            "3. Zero-Cost Rule: Transportation, crane handling, and repair of government distribution "
            "transformers are borne entirely by KPDCL. Citizens must NOT pay any private transport or repair fee.\n"
            "4. Escalation: If restoration exceeds 48 hours in rural areas, the matter escalates directly to "
            "Executive Engineer (XEN) KPDCL Division Kulgam and Deputy Commissioner's Control Room."
        )
    },
    {
        "sop_id": "SOP-RDD-PMAYG-2025-02",
        "title": "PMAY-G Beneficiary Verification & Exclusion Grievance SOP",
        "content": (
            "1. Eligibility: PMAY-G assistance is strictly restricted to houseless families or families living in "
            "zero-, one-, or two-room kutcha houses listed in the SECC-2011 / Awaas+ Permanent Waitlist (PWL).\n"
            "2. Installment Release: Assistance of Rs. 1.30 lakh (for UT of J&K hilly areas) is released in "
            "3 DBT installments linked to mandatory geo-tagged photographs on the AwaasSoft app (Sanction, Plinth, Completion).\n"
            "3. Wrongful Allotment Inquiry: If a citizen complains that a wealthy/ineligible household received "
            "a PMAY-G shed, the Block Development Officer (BDO) must conduct a spot inquiry within 10 working days "
            "and submit a report to ACD Kulgam for recovery if fraud is confirmed."
        )
    },
    {
        "sop_id": "SOP-IFC-KOUL-2025-03",
        "title": "Irrigation Koul Anti-Encroachment & Desilting SOP",
        "content": (
            "1. Joint Demarcation: Upon receiving an encroachment complaint on an Irrigation Koul, XEN Irrigation & "
            "Flood Control (I&FC) must request joint demarcation from the concerned Tehsildar within 7 days.\n"
            "2. Impartial Removal: Encroachment removal drives must cover the entire demarcated reach of the koul "
            "uniformly without selective exemption; officers found conducting selective removal face disciplinary inquiry.\n"
            "3. Pre-Sowing Desilting: Routine desilting of irrigation channels must be completed before April 15 each year."
        )
    },
    {
        "sop_id": "SOP-ADM-GFR-2025-04",
        "title": "GFR 2017 & GeM Procurement Violation Inquiry SOP",
        "content": (
            "1. Mandatory Tendering: Hiring of private vehicles or purchase of goods above Rs. 50,000 must follow "
            "GeM portal norms or competitive e-tendering under General Financial Rules (GFR) 2017.\n"
            "2. Inquiry Authority: Any complaint alleging bypassing of GFR norms or arbitrary rejection of genuine bids "
            "is forwarded to ADC / ACR Kulgam or District Vigilance Officer, who must freeze pending payments and "
            "submit a preliminary fact-finding report within 15 days."
        )
    },
    {
        "sop_id": "SOP-DMRRR-JK-05",
        "title": "DMRRR J&K Natural Calamity Relief Rates (SDRF) — Human Life, Injury, House & Household Damage",
        "content": (
            "Applies to notified natural calamities (flood, landslide, cloudburst, fire, earthquake, avalanche, hailstorm, cold wave) under DMRRR J&K / SDRF norms:\n"
            "1. Loss of Human Life: Ex-gratia of Rs. 4,00,000 (Rs. 4.00 lakh) per deceased person.\n"
            "2. Permanent Disability (Loss of limb/eye): Rs. 74,000 per person for 40% to 60% disability; Rs. 2,50,000 (Rs. 2.50 lakh) per person for disability above 60% (certified by Govt doctor).\n"
            "3. Grievous Injury: Rs. 16,000 per person hospitalized for more than a week; Rs. 5,400 per person hospitalized for less than a week.\n"
            "4. Clothing & Household Utensils (house washed away/fully damaged/inundated over 2 days): Rs. 2,500 per family for clothing and Rs. 2,500 per family for utensils/household goods.\n"
            "5. Fully/Severely Damaged House (Pucca or Kutcha): Rs. 1,30,000 per house in hilly areas (including District Kulgam) and Rs. 1,20,000 per house in plain areas.\n"
            "6. Partially Damaged House (at least 15% damage): Rs. 6,500 per Pucca house and Rs. 4,000 per Kutcha house.\n"
            "7. Damaged/Destroyed Hut: Rs. 8,000 per hut. Cattle shed attached with house: Rs. 3,000 per shed.\n"
            "8. Verification & Disbursement: Joint verification by Tehsildar/Revenue and PWD/R&B is mandatory; relief is credited via DBT/PFMS within 15 days."
        )
    },
    {
        "sop_id": "SOP-DMRRR-JK-06",
        "title": "DMRRR J&K Natural Calamity Relief Rates (SDRF) — Agriculture, Horticulture Crops & Land Desilting",
        "content": (
            "Applies to farmers and orchardists suffering 33% or more crop loss due to hailstorm, flood, unseasonal snow, drought, or pest attack under DMRRR J&K / SDRF norms (ceiling of 2 hectares per farmer):\n"
            "1. Rainfed Agriculture/Annual Crops: Rs. 8,500 per hectare (minimum Rs. 1,000 per farmer).\n"
            "2. Assured Irrigated Crops: Rs. 17,000 per hectare (minimum Rs. 2,000 per farmer).\n"
            "3. Perennial Crops & Apple Orchards / Agro-forestry: Rs. 22,500 per hectare (minimum Rs. 2,500 per farmer).\n"
            "4. Sericulture: Rs. 6,000 per hectare (Eri/Mulberry/Tussar) and Rs. 7,500 per hectare (Muga), minimum Rs. 1,000 per farmer.\n"
            "5. De-silting of Agricultural Land (sand/silt deposit thicker than 3 inches): Rs. 18,000 per hectare (minimum Rs. 2,200 per farmer).\n"
            "6. Note: Assistance is adjusted against any PMFBY crop insurance claim already received for the same calamity."
        )
    },
    {
        "sop_id": "SOP-DMRRR-JK-07",
        "title": "DMRRR J&K Natural Calamity Relief Rates (SDRF) — Livestock, Poultry & Artisans Damage",
        "content": (
            "Applies to small/marginal farmers, landless livestock owners, and artisans affected by natural calamity under DMRRR J&K / SDRF norms:\n"
            "1. Large Milch Animals (Cow, Buffalo, Camel, Yak): Rs. 37,500 per animal (ceiling of 3 large milch animals).\n"
            "2. Small Milch/Meat Animals (Sheep, Goat, Pig): Rs. 4,000 per animal.\n"
            "3. Large Draught Animals (Horse, Bullock, Camel): Rs. 32,000 per animal.\n"
            "4. Small Draught / Young Stock (Calf, Donkey, Pony, Mule, Heifer): Rs. 20,000 per animal.\n"
            "5. Poultry Loss: Rs. 100 per bird, subject to a maximum ceiling of Rs. 10,000 per household.\n"
            "6. Artisans Assistance: Rs. 5,000 per artisan for replacement of damaged tools/equipment and Rs. 5,000 per artisan for loss of raw material or finished goods (certified by competent authority)."
        )
    }
]

sop_texts = [f"{doc['title']} {doc['content']}" for doc in SOP_KNOWLEDGE_BASE]
rag_vectorizer = TfidfVectorizer(stop_words="english")
sop_matrix = rag_vectorizer.fit_transform(sop_texts)

RAG_SYSTEM_PROMPT = """
You are a Policy-Compliant Government Assistant for District Kulgam.
STRICT GROUNDING RULES:
1. Answer the citizen's query USING ONLY the facts explicitly written inside the [RETRIEVED SOP] block.
2. Mention the exact SOP ID (e.g., SOP-KPDCL-2025-01) in your answer.
3. Do NOT invent, assume, or add any external rules, compensation amounts, or timelines not stated in [RETRIEVED SOP].
4. If the [RETRIEVED SOP] does NOT contain the answer, reply ONLY with:
   "This specific detail is not covered in the provided SOP guidelines. Please consult the concerned District Officer for clarification."
""".strip()


def retrieve_relevant_sop(query: str, threshold: float = 0.05):
    query_vec = rag_vectorizer.transform([query])
    scores = cosine_similarity(query_vec, sop_matrix).flatten()
    best_idx = int(np.argmax(scores))
    best_score = float(scores[best_idx])
    if best_score < threshold:
        return None, best_score
    return SOP_KNOWLEDGE_BASE[best_idx], best_score


def answer_with_mini_rag_and_pii_masking(citizen_query: str):
    """Masks PII first, then runs TF-IDF SOP retrieval + grounded LLM generation on masked text."""
    if not citizen_query or not citizen_query.strip():
        return "", "No query provided.", "Please enter a policy or scheme question."

    # STEP 1: Mandatory PII Masking BEFORE Retrieval and LLM
    sanitized_query, audit_summary = mask_pii_asterisks(citizen_query.strip())
    masked_display = f"{sanitized_query}\n\n[{audit_summary}]"

    # STEP 2: Retrieve SOP using sanitized query
    sop_doc, score = retrieve_relevant_sop(sanitized_query)

    if sop_doc is None:
        return (
            masked_display,
            f"No matching SOP found (max similarity: {score:.2f}).",
            "This specific detail is not covered in the provided SOP guidelines. Please consult the concerned District Officer for clarification."
        )

    retrieved_display = (
        f"**SOP ID:** {sop_doc['sop_id']}\n"
        f"**Title:** {sop_doc['title']}\n"
        f"**Retrieval Similarity Score:** {score:.2f}\n\n"
        f"**Policy Text Passed to LLM:**\n{sop_doc['content']}"
    )

    augmented_prompt = (
        f"[RETRIEVED SOP]\n"
        f"SOP ID: {sop_doc['sop_id']}\n"
        f"Title: {sop_doc['title']}\n"
        f"Guidelines:\n{sop_doc['content']}\n"
        f"[/RETRIEVED SOP]\n\n"
        f"Citizen Query: {sanitized_query}\n\n"
        f"Provide a concise, strictly grounded answer:"
    )

    grounded_answer = generate_local_llm(
        system_prompt=RAG_SYSTEM_PROMPT,
        user_prompt=augmented_prompt,
        max_new_tokens=220,
        deterministic=True
    )

    return masked_display, retrieved_display, grounded_answer

print("Privacy-preserved Task 3.1 and Task 3.2 handlers ready.")

Privacy-preserved Task 3.1 and Task 3.2 handlers ready.


In [4]:
# Dynamically build synthetic 12-digit test IDs at runtime for Gradio demo buttons
def _make_dummy_12_digit():
    return f"{random.randint(2000, 9999)} {random.randint(1000, 9999)} {random.randint(1000, 9999)}"

sample_id_1 = _make_dummy_12_digit()
sample_id_2 = _make_dummy_12_digit()

with gr.Blocks(title="Privacy-Preserving CPGRAMS AI Co-Pilot — District Kulgam") as demo:
    gr.Markdown(
        "# Privacy-Preserving CPGRAMS Officer Co-Pilot (District Kulgam)\n"
        f"**Local Model:** `{MODEL_ID}` on `{device.upper()}` | "
        "**Security Gate:** Automatic Asterisk PII Masking (`****`) active before every LLM call"
    )

    # ---------------------------------------------------------
    # TAB 1: Task 3.1 (C.R.O.S.S. Triage + Live PII Masking)
    # ---------------------------------------------------------
    with gr.Tab("Task 3.1: Complaint Triage + PII Masking"):
        gr.Markdown(
            "Enter a raw complaint (even with Aadhaar, phone, email, or home address). "
            "The system automatically masks PII with asterisks **before** sending text to the LLM."
        )
        complaint_input = gr.Textbox(
            label="Raw Citizen Complaint (Unmasked Input)",
            placeholder="Paste raw citizen complaint with phone/Aadhaar/address to test live masking...",
            lines=5
        )
        triage_btn = gr.Button("Mask PII & Generate Co-Pilot Output", variant="primary")

        # Using plain Textbox (not Markdown) so asterisks (****) display literally without bold formatting
        masked_preview = gr.Textbox(
            label="Sanitized Text Actually Sent to LLM (PII Masked)",
            interactive=False,
            lines=4
        )
        audit_log_box = gr.Textbox(
            label="Privacy Audit Log",
            interactive=False,
            lines=1
        )
        triage_output = gr.Markdown(label="Officer Co-Pilot Output")

        gr.Examples(
            examples=[
                [
                    f"This is ridiculous!! I am Tariq Ahmad, R/o House No. 14, Village Bugam, Yaripora. "
                    f"Our 100 KVA transformer burnt 9 days ago! My Aadhaar is {sample_id_1} and mobile "
                    f"is +91-9876543210 (email: tariq.kulgam@example.com). Fix it now or we block the road!!"
                ],
                [
                    f"We are a poor BPL family, Resident of Ward 4, Manzgam D.H. Pora. Still no PMAY-G "
                    f"shed allotted while rich families got funds! Contact me on 9419012345, Aadhaar {sample_id_2}."
                ]
            ],
            inputs=complaint_input
        )

        triage_btn.click(
            fn=analyze_grievance_with_pii_masking,
            inputs=complaint_input,
            outputs=[masked_preview, audit_log_box, triage_output]
        )

    # ---------------------------------------------------------
    # TAB 2: Task 3.2 (Mini-RAG + Live PII Masking)
    # ---------------------------------------------------------
    with gr.Tab("Task 3.2: Mini-RAG SOP Assistant + PII Masking"):
        gr.Markdown(
            "Citizen policy queries are also stripped of PII with asterisks before SOP retrieval and LLM generation."
        )
        rag_query = gr.Textbox(
            label="Citizen Policy / Scheme Query (Unmasked Input)",
            placeholder="Ask about KPDCL timelines, PMAY-G inquiry, Irrigation Koul, or GFR norms...",
            lines=3
        )
        rag_btn = gr.Button("Mask PII, Retrieve SOP & Answer", variant="primary")

        rag_masked_preview = gr.Textbox(
            label="Sanitized Query Sent to Retriever & LLM",
            interactive=False,
            lines=3
        )

        with gr.Row():
            retrieved_sop_box = gr.Markdown(label="Retrieved SOP Source (Audit Trail)")
            rag_answer_box = gr.Markdown(label="Strictly Grounded LLM Response")

        gr.Examples(
            examples=[
                [
                    f"My name is Bilal, R/o House No. 8, Manzgam D.H. Pora (Phone: 9906123456, Aadhaar: {sample_id_1}). "
                    f"Within how many hours must KPDCL replace our rural transformer, and do villagers pay for crane transport?"
                ],
                [
                    f"Call me at +91-9419554433. A rich family got a PMAY-G house in our village. "
                    f"Who conducts the inquiry and within how many working days?"
                ],
                [
                    "Because of a 5-day transformer failure in Yaripora, my refrigerator food spoiled and TV broke. "
                    "How much cash compensation will KPDCL pay for my TV?"
                ]
            ],
            inputs=rag_query
        )

        rag_btn.click(
            fn=answer_with_mini_rag_and_pii_masking,
            inputs=rag_query,
            outputs=[rag_masked_preview, retrieved_sop_box, rag_answer_box]
        )

demo.launch(inline=True, share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>